In [2]:
import os
import torch
import torchvision
from torch.utils.data import DataLoader, Subset, Dataset
import random
import sys
from pathlib import Path
from PIL import Image

import annoy
import json

sys.path.insert(0, os.path.abspath(".."))

# Project imports
# from local_datasets import get_dataset
# from utils.helper import load_config
import matplotlib.pyplot as plt


import torch
import numpy as np
from sklearn.decomposition import PCA
import scipy
from tqdm import tqdm

from torchvision import transforms


random.seed(73)
torch.manual_seed(73)
CELEBA_MEAN = [0.5, 0.5, 0.5]
CELEBA_STD = [0.5, 0.5, 0.5]

# Root directory for the dataset
celeba_root = Path('/BS/databases08/CelebA')
# Pick one: 'all', 'train', 'valid', 'test'
celeba_split = 'all'
# CelebA image folder and partition file under the root
celeba_image_dir = celeba_root / 'img_align_celeba'
celeba_partition_file = celeba_root / 'Anno' / 'list_eval_partition.txt'
# Spatial size of training images, images are resized to this size.
image_size = 64

# -----------------------------------------------------------------------------
# Old loader code kept for reference (commented as requested)
# -----------------------------------------------------------------------------
# data_root = '/BS/databases08/CelebA'
# image_size = 64
# celeba_data = torchvision.datasets.ImageFolder(root=data_root, transform=transforms.Compose([
#                                   transforms.Resize(image_size),
#                                   transforms.CenterCrop(image_size),
#                                   transforms.ToTensor(),
#                                   transforms.Normalize(mean=CELEBA_MEAN,
#                                                        std=CELEBA_STD)
#                               ]))
#
# celeba_loader = DataLoader(
#     celeba_data, batch_size=64, shuffle=False,
#     num_workers=4, pin_memory=True
# )
#
# print(celeba_data)


class CelebAImages(Dataset):
    def __init__(self, image_dir, partition_file=None, split='all', transform=None):
        self.image_dir = Path(image_dir)
        self.partition_file = Path(partition_file) if partition_file is not None else None
        self.split = str(split).lower()
        self.transform = transform
        self.samples = self._build_samples()

    def _build_samples(self):
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
        all_paths = sorted(p for p in self.image_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts)
        if not all_paths:
            raise RuntimeError(f'No image files found in {self.image_dir}')

        if self.split == 'all' or self.partition_file is None or not self.partition_file.exists():
            return [(str(p), idx) for idx, p in enumerate(all_paths)]

        partition_map = {}
        for line in self.partition_file.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            file_name, split_id = line.split()
            partition_map[file_name] = int(split_id)

        split_ids = {'train': 0, 'valid': 1, 'val': 1, 'test': 2}
        if self.split not in split_ids:
            raise ValueError("split must be one of: 'all', 'train', 'valid', 'test'")
        wanted = split_ids[self.split]
        filtered = [p for p in all_paths if partition_map.get(p.name) == wanted]
        if not filtered:
            raise RuntimeError(f'No images found for split={self.split} in {self.partition_file}')
        return [(str(p), idx) for idx, p in enumerate(filtered)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label


celeba_data = CelebAImages(
    image_dir=celeba_image_dir,
    partition_file=celeba_partition_file,
    split=celeba_split,
    transform=transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=CELEBA_MEAN, std=CELEBA_STD)
    ])
)

celeba_loader = DataLoader(
    celeba_data, batch_size=64, shuffle=False,
    num_workers=4, pin_memory=True
)

print(f'CelebA root:   {celeba_root}')
print(f'Image dir:     {celeba_image_dir}')
print(f'Partition file:{celeba_partition_file}')
print(f'Split:         {celeba_split}')
print(f'Loaded items:  {len(celeba_data)}')


CelebA root:   /BS/databases08/CelebA
Image dir:     /BS/databases08/CelebA/img_align_celeba
Partition file:/BS/databases08/CelebA/Anno/list_eval_partition.txt
Split:         all
Loaded items:  202599


## Build kNN Index


In [3]:
from pathlib import Path
import json

# Prefer prebuilt project index to avoid expensive notebook-side rebuild.
REPO_ROOT = Path.cwd().resolve().parent
PREBUILT_INDEX_DIR = REPO_ROOT / 'output' / 'smile_classification' / 'celeba' / 'index' / 'pixel' / 'annoy' / 'euclidean'
PREBUILT_INDEX_PATH = PREBUILT_INDEX_DIR / 'index.ann'
PREBUILT_META_PATH = PREBUILT_INDEX_DIR / 'metadata.json'

if PREBUILT_INDEX_PATH.exists():
    print('Prebuilt index found. Skipping rebuild cell.')
    print(f'Index path: {PREBUILT_INDEX_PATH}')
    if PREBUILT_META_PATH.exists():
        with open(PREBUILT_META_PATH, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        print('Metadata summary:')
        print(f"  dataset: {meta.get('dataset', 'n/a')}")
        print(f"  image_size: {meta.get('image_size', 'n/a')}")
        print(f"  embedding_dim: {meta.get('embedding_dim', meta.get('dim', meta.get('vector_dim', 'n/a')))}")
else:
    print('Prebuilt index not found at expected path.')
    print('If you need to build locally, run the old build code manually.')
    print(f'Expected: {PREBUILT_INDEX_PATH}')

Prebuilt index found. Skipping rebuild cell.
Index path: /BS/dniazi_thesis/work/RandomizedSmothingmanifold/output/smile_classification/celeba/index/pixel/annoy/euclidean/index.ann
Metadata summary:
  dataset: CelebA
  image_size: 224
  embedding_dim: 150528


In [ ]:
from pathlib import Path
import json

# Load prebuilt Annoy index from project outputs instead of local notebook file.
REPO_ROOT = Path.cwd().resolve().parent
INDEX_DIR = REPO_ROOT / 'output' / 'smile_classification' / 'celeba' / 'index' / 'pixel' / 'annoy' / 'euclidean'
INDEX_PATH = INDEX_DIR / 'index.ann'
META_PATH = INDEX_DIR / 'metadata.json'

if not INDEX_PATH.exists():
    raise FileNotFoundError(
        f'Could not find prebuilt index at: {INDEX_PATH}\n'
        'Build it first or set INDEX_PATH to your existing `.ann` file.'
    )

metadata = {}
if META_PATH.exists():
    with open(META_PATH, 'r', encoding='utf-8') as f:
        metadata = json.load(f)

vector_dim = int(
    metadata.get('embedding_dim', metadata.get('dim', metadata.get('vector_dim', 64 * 64 * 3)))
)
meta_image_size = metadata.get('image_size', None)

if meta_image_size is not None:
    index_h = int(meta_image_size)
    index_w = int(meta_image_size)
else:
    side = int(round(np.sqrt(vector_dim / 3)))
    if side * side * 3 != vector_dim:
        raise ValueError(
            f'Cannot infer image shape from vector_dim={vector_dim}. '
            'Provide metadata.image_size in metadata.json.'
        )
    index_h = side
    index_w = side

index_shape_chw = (3, index_h, index_w)

knn_index = annoy.AnnoyIndex(vector_dim, 'euclidean')
knn_index.load(str(INDEX_PATH))

print(f'Loaded Annoy index: {INDEX_PATH}')
print(f'Vector dim: {vector_dim}')
print(f'Index image size: {index_h}x{index_w}')
if int(np.prod(celeba_data[0][0].shape)) != vector_dim:
    print('Note: dataset transform shape differs from index embedding shape.')
    print(f'  dataset tensor shape: {tuple(celeba_data[0][0].shape)}')
    print(f'  index tensor shape:   {index_shape_chw}')

scale_weight = 0.7


def unnormalize_np(img):
    return np.add(np.multiply(img, CELEBA_STD), CELEBA_MEAN)


target_idcs = [100, 51, 42]
analysis_cache_from_cell4 = []

for target_idx in target_idcs:

    X_img, X_class = celeba_data[target_idx]  # sample for visualization reference
    k = 500
    X_nns_idcs = knn_index.get_nns_by_item(target_idx, k)

    X_local_n = []
    for i in X_nns_idcs:
        X_local_n.append(knn_index.get_item_vector(i))
    X_local_n = np.asarray(X_local_n, dtype=np.float64)
    print(X_local_n.shape)

    # original image from index embedding space
    X_orig_flat = np.asarray(knn_index.get_item_vector(target_idx), dtype=np.float64)
    X_resh = np.reshape(X_orig_flat, index_shape_chw)
    orig_img = unnormalize_np(np.transpose(X_resh, (1, 2, 0)))

    ## Get mean and recenter
    mean_local_n = np.mean(X_local_n, 0)
    X_local_n = X_local_n - mean_local_n

    ## Compute PCA space for the neighborhood
    n_components = min(k, X_local_n.shape[0], X_local_n.shape[1])
    pca = PCA(n_components=n_components)
    pca.fit(X_local_n)
    ev = pca.explained_variance_
    Vt = pca.components_
    ev_norm = np.maximum(ev, 1e-12) / np.max(np.maximum(ev, 1e-12))

    anchor_2d = np.matmul(X_orig_flat - mean_local_n, np.transpose(Vt[:2]))
    neigh_2d = np.matmul(X_local_n, np.transpose(Vt[:2]))
    analysis_cache_from_cell4.append({
        'target_idx': int(target_idx),
        'evals': np.asarray(ev, dtype=np.float64),
        'evals_norm': np.asarray(ev_norm, dtype=np.float64),
        'anchor_2d': np.asarray(anchor_2d, dtype=np.float64),
        'neigh_2d': np.asarray(neigh_2d, dtype=np.float64),
        'k_available': int(len(ev_norm)),
    })

    X_whitened = np.matmul(X_orig_flat, (1 / np.sqrt(ev)) * np.transpose(Vt))

    X_resh = np.matmul(X_whitened, np.transpose(np.sqrt(ev) * np.transpose(Vt)))
    X_resh = np.reshape(X_resh, index_shape_chw)
    orig_rec = unnormalize_np(np.transpose(X_resh, (1, 2, 0)))

    n_samples = 5
    fig, axes = plt.subplots(4, 5, figsize=(20, 20))
    for ax in axes.flat:
        ax.axis('off')

    axes[0, 0].imshow(orig_img)
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    axes[0, 1].imshow(orig_rec)
    axes[0, 1].set_title('PCA Reconstruction')
    axes[0, 1].axis('off')

    # noisy samples in whitened space
    for i in range(n_samples):
        X_noised = X_whitened + np.random.normal(0.0, scale_weight**2, size=n_components)

        X_noised = np.matmul(X_noised, np.transpose(np.sqrt(ev) * np.transpose(Vt))) + mean_local_n
        rec_img = np.transpose(np.reshape(X_noised, index_shape_chw), (1, 2, 0))
        rec_img = unnormalize_np(rec_img)
        ax = axes[(i // 5) + 1, i % 5]
        if i == 0:
            ax.set_title('Samples with noise in whitened space')
        ax.imshow(rec_img)
        ax.axis('off')

    # noisy samples in original space
    for i in range(n_samples):
        X_noised = X_orig_flat + np.random.normal(0.0, scale_weight**2, size=X_orig_flat.shape)
        rec_img = np.transpose(np.reshape(X_noised, index_shape_chw), (1, 2, 0))
        rec_img = unnormalize_np(rec_img)
        ax = axes[((i + 1 * n_samples) // 5) + 1, (i + 1 * n_samples) % 5]
        if i == 0:
            ax.set_title('Samples with noise in original space')
        ax.imshow(rec_img)
        ax.axis('off')

    for i, target_i in enumerate(X_nns_idcs[0:5]):
        X_orig_flat = np.asarray(knn_index.get_item_vector(target_i), dtype=np.float64)
        X_resh = np.reshape(X_orig_flat, index_shape_chw)
        neigh_img = unnormalize_np(np.transpose(X_resh, (1, 2, 0)))
        ax = axes[((i + 2 * n_samples) // 5) + 1, (i + 2 * n_samples) % 5]
        if i == 0:
            ax.set_title('Neighbors')
        ax.imshow(neigh_img)
        ax.axis('off')

    scale_single_weight = scale_weight / 2

print(f'Cached PCA geometry for {len(analysis_cache_from_cell4)} targets (from Cell 4).')



- Isotropic region radius: $r=\sigma$
- Normalized eigenvalues: $\tilde\lambda_i=\lambda_i/\lambda_{\max}$
- Manifold axis lengths: $a_i=\sigma\sqrt{\tilde\lambda_i}$
- Geometry-only volume relation at same $\sigma$:
  $\log V_{mani,geo}-\log V_{iso,geo}=\frac12\sum_i\log(\tilde\lambda_i)$

In [ ]:

# Show cached neighborhoods from Cell 4: isotropic ball (circle) vs manifold ellipse.
# 3-panel view per sample:
#   Panel A: full neighbor cloud  (circle may be invisible dot — that's expected)
#   Panel B: mid-zoom = ±10% of PC1 cloud range  (see circle in context of nearby neighbors)
#   Panel C: tight zoom = ±3σ  (circle and ellipse clearly fill the frame)

from matplotlib.patches import Ellipse

if 'analysis_cache_from_cell4' not in globals() or len(analysis_cache_from_cell4) == 0:
    raise RuntimeError('Run Cell 4 first (it builds analysis_cache_from_cell4).')

sigma = float(globals().get('scale_weight', 0.7))
rows = []

for d in analysis_cache_from_cell4:
    anchor_2d = np.asarray(d['anchor_2d'], dtype=np.float64)
    neigh_2d  = np.asarray(d['neigh_2d'],  dtype=np.float64)
    evals     = np.asarray(d['evals'],     dtype=np.float64)

    eval_max   = float(np.max(evals)) if evals.size > 0 else 1.0
    evals_norm = np.maximum(evals, 1e-12) / eval_max
    a1 = float(sigma * np.sqrt(evals_norm[0]))   # always = sigma
    a2 = float(sigma * np.sqrt(evals_norm[1]))

    pc1_std = float(np.std(neigh_2d[:, 0]))
    pc2_std = float(np.std(neigh_2d[:, 1]))

    rows.append({
        'target_idx':    int(d['target_idx']),
        'sigma':         sigma,
        'a1 (=sigma)':   round(a1, 5),
        'a2':            round(a2, 5),
        'a2/a1':         round(a2 / (a1 + 1e-12), 5),
        'pc1_std':       round(pc1_std, 4),
        'sigma/pc1_std': round(sigma / (pc1_std + 1e-12), 6),
    })

    # ── 3 zoom levels ──────────────────────────────────────────────────────
    pc1_range = float(np.max(neigh_2d[:, 0]) - np.min(neigh_2d[:, 0]))
    zoom_mid  = 0.10 * pc1_range          # 10 % of full cloud
    zoom_tight = 3.0 * a1                  # ±3σ — always shows circle clearly

    zoom_labels = [
        (None,       f"[A] Full cloud\nPC1 std={pc1_std:.1f},  σ={sigma}  →  σ/std={sigma/pc1_std:.4f}\n"
                     f"Circle is {'VISIBLE' if sigma/pc1_std > 0.05 else 'a tiny dot (expected)'}"),
        (zoom_mid,   f"[B] Mid-zoom  ±{zoom_mid:.1f}\n(10 % of cloud)\na1={a1:.4f}, a2={a2:.4f}"),
        (zoom_tight, f"[C] Tight zoom  ±{zoom_tight:.4f}  (=3σ)\nCircle and ellipse fill the view\na2/a1={a2/a1:.4f}"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor='white')
    fig.suptitle(f"Target {d['target_idx']}  —  CelebA pixel PCA-2D geometry  (σ={sigma})", fontsize=12, y=1.02)

    for ax, (zr, title) in zip(axes, zoom_labels):
        ax.scatter(neigh_2d[:, 0], neigh_2d[:, 1], s=5, alpha=0.22, color='#4c78a8', linewidths=0, zorder=1)
        ax.add_patch(plt.Circle(
            (anchor_2d[0], anchor_2d[1]), sigma,
            fill=False, edgecolor='tab:blue', linewidth=2.5,
            linestyle=(0, (4, 2)), alpha=0.95, zorder=3,
            label=f'Iso circle  r=σ={sigma}'
        ))
        ax.add_patch(Ellipse(
            (anchor_2d[0], anchor_2d[1]),
            width=2.0 * a1, height=2.0 * a2,
            fill=False, edgecolor='tab:orange', linewidth=2.5,
            linestyle='solid', alpha=0.95, zorder=3,
            label=f'Mani ellipse  a1={a1:.4f}, a2={a2:.4f}'
        ))
        ax.scatter(anchor_2d[0], anchor_2d[1], s=160, marker='*',
                   c='black', edgecolors='white', linewidths=1.0, zorder=5, label='Anchor')

        if zr is not None:
            ax.set_xlim(anchor_2d[0] - zr, anchor_2d[0] + zr)
            ax.set_ylim(anchor_2d[1] - zr, anchor_2d[1] + zr)
        # else: matplotlib auto-fits to all points (full cloud)

        ax.set_aspect('equal')
        ax.set_title(title, fontsize=8.5)
        ax.set_xlabel(f'PC1  (cloud std={pc1_std:.2f})')
        ax.set_ylabel(f'PC2  (cloud std={pc2_std:.2f})')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7, loc='upper right')

    plt.tight_layout()
    plt.show()

vals_df = pd.DataFrame(rows)
display(vals_df)

print('\nKey:')
print('  Panel A = full cloud.  If sigma/pc1_std << 1, circle is invisible — correct, sigma IS tiny in pixel space.')
print('  Panel B = mid-zoom shows circle in context of nearby neighbors.')
print('  Panel C = tight zoom (±3σ) always shows the shapes regardless of scale.')
print('  a1=sigma always. The squashing is a2 < a1.')


In [ ]:
# Lightweight volume calculations from Cell 4 cache (sample-focused, kernel-safe).

import math
import pandas as pd

if 'analysis_cache_from_cell4' in globals() and len(analysis_cache_from_cell4) > 0:
    analysis_cache_for_volume = analysis_cache_from_cell4
elif 'analysis_cache' in globals() and len(analysis_cache) > 0:
    analysis_cache_for_volume = analysis_cache
else:
    raise RuntimeError('Run Cell 4 first (or Cell 6) to provide cached PCA geometry.')

# ---------- controls ----------
# Choose which cached samples to compute (by position in cache).
SAMPLE_POSITIONS = [0, 1, 2]   # e.g., [0] for only first sample

# Use your smoothing sigma from Cell 4 by default; can also pass a list.
default_sigma = float(globals().get('scale_weight', 0.7))
SIGMAS_VOL = [default_sigma]    # e.g., [0.12, 0.24, 0.36]

# Optional: cap dimensions for extra safety (None = use all available).
MAX_K_DIM = None

# Optional: plotting off by default to reduce kernel pressure.
PLOT_VOLUME_SUMMARY = False
# -----------------------------

selected = []
for pos in SAMPLE_POSITIONS:
    if 0 <= int(pos) < len(analysis_cache_for_volume):
        selected.append(analysis_cache_for_volume[int(pos)])

if len(selected) == 0:
    raise ValueError(f'No valid SAMPLE_POSITIONS in range [0, {len(analysis_cache_for_volume)-1}]')

rows = []
for d in selected:
    evals_norm_full = np.asarray(d['evals_norm'], dtype=np.float64)
    k_available = int(d['k_available'])
    k = k_available if MAX_K_DIM is None else int(min(k_available, MAX_K_DIM))

    if k < 1:
        continue

    lam = np.maximum(evals_norm_full[:k], 1e-30)

    # Geometry-only shape term (independent of sigma).
    log_shape = 0.5 * float(np.sum(np.log(lam)))
    anisotropy = float(np.sqrt(lam[0] / lam[-1])) if k >= 2 else 1.0

    # log volume of k-ball constant C_k
    log_unit_ball = (k / 2.0) * np.log(np.pi) - math.lgamma(k / 2.0 + 1.0)

    for sigma in SIGMAS_VOL:
        sigma = float(sigma)
        if sigma <= 0:
            continue

        log_v_iso = float(log_unit_ball + k * np.log(sigma))
        log_v_mani = float(log_v_iso + log_shape)

        rows.append({
            'target_idx': int(d['target_idx']),
            'cache_pos': int(analysis_cache_for_volume.index(d)),
            'sigma': sigma,
            'k_dim': int(k),
            'log_v_iso': log_v_iso,
            'log_v_mani': log_v_mani,
            'log_geo_ratio': float(log_shape),
            'geo_ratio': float(np.exp(np.clip(log_shape, -700, 700))),
            'anisotropy': anisotropy,
        })

vol_df = pd.DataFrame(rows).sort_values(['target_idx', 'sigma']).reset_index(drop=True)
display(vol_df.round(6))

vol_mean = (
    vol_df.groupby(['sigma'], as_index=False)
          .agg(
              mean_k_dim=('k_dim', 'mean'),
              mean_log_geo_ratio=('log_geo_ratio', 'mean'),
              mean_anisotropy=('anisotropy', 'mean')
          )
)
display(vol_mean.round(6))

if PLOT_VOLUME_SUMMARY and len(vol_mean) > 0:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor='white')

    axes[0].plot(vol_mean['sigma'], vol_mean['mean_log_geo_ratio'], marker='o', linewidth=2, color='tab:blue')
    axes[0].set_title('Mean log geometry ratio: log(V_mani_geo / V_iso_geo)')
    axes[0].set_xlabel('sigma')
    axes[0].set_ylabel('mean log ratio')
    axes[0].grid(alpha=0.3)

    axes[1].plot(vol_mean['sigma'], vol_mean['mean_anisotropy'], marker='o', linewidth=2, color='tab:orange')
    axes[1].set_title('Mean anisotropy across selected samples')
    axes[1].set_xlabel('sigma')
    axes[1].set_ylabel('mean anisotropy')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

print(f'Done: volume metrics computed for {len(selected)} sample(s), sigmas={SIGMAS_VOL}, MAX_K_DIM={MAX_K_DIM}.')